# Doc Anchor AI: OCR & INGESTION EVALUATION (COLAB GPU)

Notebook này được thiết kế để đánh giá toàn trình (End-to-End) Ingestion Pipeline (PaddleOCR + Qwen2-VL local qua Ollama) của dự án FinSight AI.

Push toàn bộ thư mục local lên Github, sau đó chạy notebook này trên Colab. Nó sẽ tự động Clone/Pull code về, cài đặt mọi thứ và tiến hành chấm điểm Benchmark.

In [ ]:
import os
REPO_URL = 'https://github.com/HoangKhang226/Doc-Anchor-AI.git'  # TODO: Thay đổi link này nếu tên repo khác
REPO_DIR = '/content/Doc-Anchor-AI'

if os.path.exists(REPO_DIR):
    %cd $REPO_DIR
    !git pull
else:
    !git clone $REPO_URL $REPO_DIR
    %cd $REPO_DIR

In [ ]:
# 1. Cập nhật hệ thống và cài đặt thư viện lõi
!apt-get update -y && apt-get install -y ffmpeg zstd poppler-utils tesseract-ocr

# 2. Cài siêu tốc bằng uv (Gom tất cả vào 1 lệnh duy nhất để tránh xung đột và Restart)
!pip install uv
!uv pip install --system \
    numpy==1.26.4 \
    paddlepaddle-gpu==2.6.2 \
    paddleocr==2.7.3 \
    pyclipper lmdb imgaug visualdl \
    -r requirements.txt


## Khởi động Ollama Server & Pull Model (Qwen2-VL)

In [ ]:
# 3. Tải và cài đặt Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 4. Khởi chạy Ollama background và ghi log
!nohup ollama serve > ollama.log 2>&1 &

# 5. Chờ 8 giây để server Ollama khởi động xong
import time
time.sleep(8)

# 6. Kéo model Qwen2-VL:7B về
!ollama pull qwen2.5vl:7b

## Chạy Đánh Giá Benchmark (Multi-threading)

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Cập nhật code mới nhất từ Github
%cd /content/Doc-Anchor-AI
!git pull

# 3. Chạy script Python ngắn để dọn dẹp và copy 60 file cũ sang Drive theo đúng cấu trúc siêu đẹp
import shutil
from pathlib import Path

backup_dir = Path("/content/drive/MyDrive/DocAnchor_Backup")
backup_dir.mkdir(parents=True, exist_ok=True)

# Chép file JSON điểm số
report_src = Path("/content/Doc-Anchor-AI/evaluation/ocr/eval_report.json")
if report_src.exists():
    shutil.copy(report_src, backup_dir / "eval_report.json")

# Chép 60 file Markdown cũ
data_dir = Path("/content/Doc-Anchor-AI/evaluation/ocr/data/categorized")
for md_file in data_dir.rglob("*.pred.md"):
    dest = backup_dir / md_file.parent.name
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copy(md_file, dest / md_file.name)
print("đã chép kết quả cũ sang Google Drive!")

# 4. Cuối cùng mới chạy lệnh Benchmark để làm nốt 40 ảnh còn lại
!python evaluation/ocr/scripts/run_ocr_eval.py --workers 3 --backup_dir "/content/drive/MyDrive/DocAnchor_Backup"